In [ ]:
using Plots, DifferentialEquations, NLsolve

In [ ]:
const I0 = 1e-12 # A
const κ = 0.7
const Vdd = 1.8 # V
const UT= 25*1e-3 # V
const C=1e-6 # F
;

\begin{align*}
    \text{M1: }& I_{cmp} = I_{0} e^{\kappa\left(V_{d d}-V_{in}\right) / U_T}\left(1-e^{-\left(V_{d d}-V_1\right) / U_T}\right)\\
    \text{M2: }& I_{1} = I_{0} e^{\kappa V_0 / U_T}\left(1-e^{-V_1 / U_T}\right)\\
    \text{M3: }& I_{lin1} = I_{0} e^{\left(\kappa V_1 - V_{out} )\right/ U_T}\left(1-e^{-\left(V_1-V_{out}\right) / U_T}\right)\\
    \text{M4: }& I_{lin2} = I_{0} e^{\left(\kappa V_{out} - V_{2} \right)/ U_T}\left(1-e^{-\left(V_{out}-V_{2}\right) / U_T}\right)\\
    \text{M5: }& I_{lin3} = I_{0} e^{\kappa V_{lin} / U_T}\left(1-e^{-V_2 / U_T}\right)\\
    \text{K1: }& C_1\dot V_1 = I_{cmp} - I_1 - I_{lin1} \\
    \text{Kout: }& C_{out} \dot V_{out} = I_{lin1} - I_{lin2} \\
    \text{K2: }& C_2\dot V_2 = I_{lin2} - I_{lin3}
\end{align*}

In [ ]:
# Static transistor equations
Icmp(Vin,V1) = I0 * exp(κ*(Vdd-Vin)/UT) * (1 - exp(-(Vdd-V1)/UT))
I1(V0,V1) = I0 * exp(κ*V0/UT) * (1 - exp(-V1/UT))
Ilin1(V1,Vout) = I0 * exp((κ*V1 - Vout)/UT) * (1 - exp(-(V1-Vout)/UT))
Ilin2(Vout,V2) = I0 * exp((κ*Vout - V2)/UT) * (1 - exp(-(Vout-V2)/UT))
Ilin3(Vlin,V2) = I0 * exp(κ*Vlin/UT) * (1 - exp(-V2/UT));

In [ ]:
function Imode_sigmoid!(du,u,p,t)
    
    Vin = p[1]
    Vlin = p[2]
    V0 = p[3]
    
    V1=u[1]
    Vout=u[2]
    V2=u[3]

    du[1]= (Icmp(Vin(t),V1) - I1(V0,V1) - Ilin1(V1,Vout))/C
    du[2]= (Ilin1(V1,Vout) - Ilin2(Vout,V2))/C
    du[3]= (Ilin2(Vout,V2) - Ilin3(Vlin,V2))/C
    
end;